# 21 · Reinforcement Learning

- **Level:** Advanced → Modern
- **Typical semester:** Semester 7–8 electives (Reinforcement Learning) and research
- **Prerequisites:** 11, 15, 18
- **Matches code in:** `lipipy.rl`


## 1. Markov decision process

Tuple $(\mathcal S, \mathcal A, P, R, \gamma)$; policy $\pi(a|s)$; return $G_t = \sum_{k=0}^\infty\gamma^kr_{t+k+1}$

$V^\pi(s) = \mathbb E_\pi[G_t|s_t = s]$, $\;Q^\pi(s,a) = \mathbb E_\pi[G_t|s_t = s, a_t = a]$, $\;A^\pi(s,a) = Q^\pi(s,a) - V^\pi(s)$


## 2. Bellman equations

**Expectation:**
$$V^\pi(s) = \sum_a\pi(a|s)\sum_{s',r}p(s',r|s,a)[r + \gamma V^\pi(s')]$$
$$Q^\pi(s,a) = \sum_{s',r}p(s',r|s,a)\left[r + \gamma\sum_{a'}\pi(a'|s')Q^\pi(s',a')\right]$$

**Optimality:**
$$V^*(s) = \max_a\sum_{s',r}p(s',r|s,a)[r + \gamma V^*(s')], \qquad Q^*(s,a) = \sum_{s',r}p(s',r|s,a)\left[r + \gamma\max_{a'}Q^*(s',a')\right]$$

Matrix form: $V^\pi = (I - \gamma P^\pi)^{-1}R^\pi$. The Bellman operator is a $\gamma$-contraction, so value iteration converges.


## 3. Dynamic programming

**Policy evaluation:** iterate the expectation equation. **Policy improvement:** $\pi'(s) = \arg\max_aQ^\pi(s,a)$. **Policy iteration** alternates them. **Value iteration:** $V_{k+1}(s) = \max_a\sum p(s',r|s,a)[r + \gamma V_k(s')]$.


## 4. Bandits

Regret $R_T = T\mu^* - \sum_t\mu_{a_t}$. **ε-greedy.** **UCB1:** $a_t = \arg\max_a\left[\hat\mu_a + \sqrt{\frac{2\ln t}{N_a}}\right]$ (regret $O(\log T)$). **Thompson sampling:** sample $\theta_a\sim$ posterior, pick $\arg\max\theta_a$ (Beta-Bernoulli: $\text{Beta}(1 + S_a, 1 + F_a)$). Incremental mean $Q_{n+1} = Q_n + \frac1n(R_n - Q_n)$.

**LinUCB:** $a_t = \arg\max_a\left[x_a^\top\hat\theta + \alpha\sqrt{x_a^\top A^{-1}x_a}\right]$


## 5. Monte Carlo and temporal-difference learning

**MC:** $V(s)\leftarrow V(s) + \alpha[G_t - V(s)]$

**TD(0):** $V(s_t)\leftarrow V(s_t) + \alpha\,\delta_t$, $\;\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)$

**$n$-step return:** $G_t^{(n)} = \sum_{k=0}^{n-1}\gamma^kr_{t+k+1} + \gamma^nV(s_{t+n})$

**TD($\lambda$):** $G_t^\lambda = (1 - \lambda)\sum_{n\ge1}\lambda^{n-1}G_t^{(n)}$; eligibility traces $e_t = \gamma\lambda e_{t-1} + \nabla V(s_t)$, $\;\theta\leftarrow\theta + \alpha\delta_te_t$

**SARSA:** $Q(s,a)\leftarrow Q + \alpha[r + \gamma Q(s', a') - Q(s,a)]$ (on-policy)

**Q-learning:** $Q(s,a)\leftarrow Q + \alpha[r + \gamma\max_{a'}Q(s',a') - Q(s,a)]$ (off-policy)

**Expected SARSA:** target $r + \gamma\sum_{a'}\pi(a'|s')Q(s',a')$. **Double Q-learning:** $r + \gamma Q_B(s', \arg\max_{a'}Q_A(s',a'))$.


## 6. Deep Q-networks

$$\mathcal L(\theta) = \mathbb E_{(s,a,r,s')\sim\mathcal D}\left[\left(r + \gamma\max_{a'}Q_{\theta^-}(s',a') - Q_\theta(s,a)\right)^2\right]$$
with replay buffer $\mathcal D$ and target network $\theta^-$.

**Double DQN:** $y = r + \gamma Q_{\theta^-}(s', \arg\max_{a'}Q_\theta(s',a'))$. **Dueling:** $Q = V(s) + A(s,a) - \frac{1}{|\mathcal A|}\sum_{a'}A(s,a')$. **Prioritized replay:** $P(i)\propto|\delta_i|^\alpha$, importance weights $(NP(i))^{-\beta}$.

**Distributional RL:** learn $Z(s,a)$ with $Z(s,a)\overset{D}{=}R + \gamma Z(S', A')$ (C51 categorical, QR-DQN quantile regression).


## 7. Policy gradient

**Policy gradient theorem:**
$$\nabla_\theta J(\theta) = \mathbb E_\pi\left[\nabla_\theta\log\pi_\theta(a|s)\,Q^\pi(s,a)\right]$$

**REINFORCE:** $\nabla J\approx\sum_t\nabla\log\pi_\theta(a_t|s_t)(G_t - b(s_t))$; a baseline $b$ reduces variance without bias.

**Actor-critic:** replace $G_t$ by $\delta_t$ or $\hat A_t$ from a learned critic $V_\phi$.

**Generalized advantage estimation (GAE):**
$$\hat A_t = \sum_{l=0}^\infty(\gamma\lambda)^l\delta_{t+l}, \qquad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

**Entropy bonus:** $+\beta H(\pi(\cdot|s))$.


## 8. Trust-region methods: TRPO and PPO

**TRPO:** maximize $\mathbb E\left[\frac{\pi_\theta(a|s)}{\pi_{old}(a|s)}\hat A\right]$ s.t. $\mathbb E[KL(\pi_{old}\|\pi_\theta)]\le\delta$ (natural gradient + line search)

**PPO-clip:**
$$L^{CLIP}(\theta) = \mathbb E_t\left[\min\left(r_t(\theta)\hat A_t,\;\text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon)\hat A_t\right)\right], \qquad r_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{old}(a_t|s_t)}$$
Full objective $L^{CLIP} - c_1L^{VF} + c_2H[\pi]$, $\epsilon\approx0.2$.


## 9. Deterministic and maximum-entropy actor-critic

**DDPG:** $\nabla_\theta J = \mathbb E[\nabla_aQ(s,a)|_{a = \mu_\theta(s)}\nabla_\theta\mu_\theta(s)]$; soft target update $\theta^-\leftarrow\tau\theta + (1-\tau)\theta^-$

**TD3:** twin critics with $\min(Q_1, Q_2)$ target, delayed policy updates, target policy smoothing.

**Soft actor-critic (SAC):** maximize $\sum_t\mathbb E[r_t + \alpha H(\pi(\cdot|s_t))]$; soft target $y = r + \gamma(\min_iQ_i(s',a') - \alpha\log\pi(a'|s'))$; policy loss $\mathbb E[\alpha\log\pi(a|s) - Q(s,a)]$; automatic temperature tuning toward a target entropy.


## 10. Model-based RL and planning

**Dyna-Q:** learn a model, do extra Q-updates on simulated transitions.

**MCTS with UCT:** select $\arg\max_a\left[Q(s,a) + c\sqrt{\frac{\ln N(s)}{N(s,a)}}\right]$; **PUCT (AlphaZero):** $Q(s,a) + c\,P(s,a)\frac{\sqrt{N(s)}}{1 + N(s,a)}$

AlphaZero loss: $(z - v)^2 - \pi^\top\log p + c\|\theta\|^2$. MuZero learns representation, dynamics and prediction functions and plans in latent space.

**World models / Dreamer:** learn a latent dynamics model (RSSM) and train the policy on imagined rollouts with $\lambda$-returns.

**Model predictive control:** at each step solve $\min_{a_{t:t+H}}\sum_k c(s_k, a_k)$ under the model, execute the first action (cross-entropy method or MPPI for sampling-based MPC).


## 11. Offline RL, imitation and inverse RL

**Behaviour cloning:** $\max\sum\log\pi(a|s)$ on expert data. **DAgger:** query the expert on states visited by the learner.

**CQL:** add $\alpha\left(\mathbb E_s\log\sum_ae^{Q(s,a)} - \mathbb E_{(s,a)\sim\mathcal D}Q(s,a)\right)$ to push down out-of-distribution Q-values.

**IQL:** expectile regression $L_2^\tau(u) = |\tau - [u < 0]|u^2$ for $V$, then advantage-weighted regression $\exp(\beta A)\log\pi$.

**Decision Transformer:** model $(\hat R_t, s_t, a_t)$ sequences with a causal transformer, conditioned on return-to-go.

**Max-entropy IRL:** $p(\tau)\propto\exp(r_\theta(\tau))$; gradient = expert feature expectations − model feature expectations. **GAIL:** a GAN between expert and policy state-action pairs.


## 12. Exploration, multi-agent and hierarchy

Count-based bonus $\frac{\beta}{\sqrt{N(s)}}$; curiosity (ICM) bonus = forward-model prediction error; RND bonus $\|f(s) - \hat f_\theta(s)\|^2$ against a fixed random network.

**Multi-agent:** Nash equilibrium; centralized training with decentralized execution (MADDPG); QMIX monotonic mixing $\frac{\partial Q_{tot}}{\partial Q_a}\ge0$.

**Options framework:** option $= (\mathcal I, \pi, \beta)$; semi-MDP Bellman equation over options.


## 13. RL for language models (bridge to 22)

Treat a prompt as the state and the generated sequence as the action; reward from a reward model or a verifier. The KL-regularized objective
$$\max_\pi\mathbb E_{x, y\sim\pi}[r(x,y)] - \beta D_{KL}(\pi(\cdot|x)\|\pi_{ref}(\cdot|x))$$
has the closed-form optimum $\pi^*(y|x) = \frac{1}{Z(x)}\pi_{ref}(y|x)\exp\left(\frac{r(x,y)}{\beta}\right)$.
